# Lecture 25 Worksheet: Diagnosing a Model (Bias, Variance, Learning Curves)
**NAVTTC / NIAI AI/ML Trainer Cohort** — works in Google Colab. Fill every `___` blank and run top to bottom; check lines print `OK`. Uses numpy only for the main tasks.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

# Invented delivery data: minutes depend on distance (km) with noise sd 2.5
SIG = 2.5
def make_data(n, seed):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.5, 9, n)
    y = 12 + 14*np.sqrt(x) + rng.normal(0, SIG, n)
    return x, y

x, y = make_data(200, seed=11)
print(x[:3], y[:3])

## Task 1: Split 60 / 20 / 20
The data are already in random order. Cut them into training, cross-validation and test sets.

In [ ]:
m = len(x)
n_tr, n_cv = int(0.6*m), int(0.2*m)
xtr, ytr = x[:___], y[:___]
xcv, ycv = x[n_tr:n_tr+n_cv], y[n_tr:n_tr+n_cv]
xte, yte = x[n_tr+n_cv:], y[n_tr+n_cv:]
print(len(xtr), len(xcv), len(xte))
assert (len(xtr), len(xcv), len(xte)) == (120, 40, 40); print("OK")

## Task 2: The error function and a polynomial fit
`J = 1/(2m) * sum((yhat - y)**2)`. Fill in the formula. The fitting helpers are given.

In [ ]:
def J(yhat, y):
    return np.sum((___ - y)**2) / (2*len(y))

def feats(x, d):                      # x, x^2, ..., x^d
    return np.vander(x, d+1, increasing=True)[:, 1:]

def fit(x, y, d, lam=0.0):
    X = feats(x, d); mu, sd = X.mean(0), X.std(0)
    A = np.c_[np.ones(len(X)), (X-mu)/sd]
    R = lam*np.eye(A.shape[1]); R[0, 0] = 0          # no penalty on b
    w = np.linalg.pinv(A.T@A + R, rcond=1e-10) @ (A.T@y)
    return w, mu, sd

def predict(model, x, d):
    w, mu, sd = model
    return np.c_[np.ones(len(x)), (feats(x, d)-mu)/sd] @ w

print(J(np.array([1.0, 2.0]), np.array([0.0, 4.0])))
assert abs(J(np.array([1.0, 2.0]), np.array([0.0, 4.0])) - 1.25) < 1e-9; print("OK")

## Task 3: Choose the degree on cross-validation error
Fit degrees 1 to 8. Print J_train, J_cv, J_test. Pick the degree with the lowest **J_cv** (not J_train!).

In [ ]:
rows = []
for d in range(1, 9):
    model = fit(xtr, ytr, d)
    rows.append((d, J(predict(model, xtr, d), ytr), J(predict(model, xcv, d), ycv), J(predict(model, xte, d), yte)))
for r in rows: print(r[0], round(r[1], 2), round(r[2], 2), round(r[3], 2))

best_d = min(rows, key=lambda r: r[___])[0]      # column 2 is J_cv
print("best degree by J_cv:", best_d)
assert 2 <= best_d <= 5 and rows[-1][1] < rows[0][1]; print("OK")

## Task 4: Bias gap and variance gap
The baseline is the noise floor. With noise sd 2.5 the best possible J is `SIG**2 / 2`. Compute both gaps for degrees 1, 2 and 8 and label them.

In [ ]:
baseline = ___                 # SIG**2 / 2
for d in (1, 2, 8):
    _, jtr, jcv, _ = rows[d-1]
    bias_gap, var_gap = jtr - baseline, jcv - jtr
    verdict = "high bias" if bias_gap > var_gap and bias_gap > 0.5 else ("high variance" if var_gap > 0.5 else "fine")
    print(d, "bias gap", round(bias_gap, 2), "variance gap", round(var_gap, 2), "->", verdict)
assert abs(baseline - 3.125) < 1e-9; print("OK")

## Task 5: Choose λ for a degree-8 model
Fit with L2 penalty strengths from 0 to 1000. Pick the λ with the lowest J_cv. (J is measured WITHOUT the penalty term.)

In [ ]:
lams = [0, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 1000]
res = []
for lam in lams:
    model = fit(xtr, ytr, 8, lam)
    res.append((lam, J(predict(model, xtr, 8), ytr), J(predict(model, xcv, 8), ycv)))
    print(lam, round(res[-1][1], 2), round(res[-1][2], 2))
best_lam = min(res, key=lambda r: r[___])[0]
print("best lambda:", best_lam)
assert 0 < best_lam < 3 and res[-1][1] > 10; print("OK: tiny lambda overfits, huge lambda underfits")

## Task 6: A learning curve
For training-set sizes 20 to 150, fit and plot J_train and J_cv. Do it for degree 1 and degree 8. One random data set is noisy, so average over 50 repeats.

In [ ]:
sizes = [20, 25, 30, 40, 55, 75, 100, 150]

def curve(d, repeats=50):
    jt = np.zeros(len(sizes)); jc = np.zeros(len(sizes))
    for r in range(repeats):
        X, Y = make_data(1000, seed=1000 + r)
        Xc, Yc = X[500:], Y[500:]                      # a large cross-validation pool
        for i, n in enumerate(sizes):
            mod = fit(X[:n], Y[:n], d)
            jt[i] += J(predict(mod, X[:n], d), Y[:n]); jc[i] += J(predict(mod, Xc, d), Yc)
    return jt/repeats, jc/repeats

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
curves = {}
for a, d in zip(ax, (1, 8)):
    jt, jc = curve(d); curves[d] = (jt, jc)
    a.plot(sizes, jc, "-o", label="J_cv"); a.plot(sizes, jt, "-o", label="J_train"); a.axhline(SIG**2/2, ls="--", c="gray")
    a.set_title(f"degree {d}"); a.set_xlabel("training examples"); a.set_ylim(0, 8)
ax[0].legend(); plt.show()

gap1 = curves[1][1][-1] - curves[1][0][-1]      # variance gap at n = 150, degree 1
gap8_small = curves[8][1][3] - curves[8][0][3]  # variance gap at n = 40, degree 8
gap8_big = curves[8][1][-1] - curves[8][0][-1]  # variance gap at n = 150, degree 8
print("deg 1 gap at 150:", round(gap1, 2), " deg 8 gap at 40:", round(gap8_small, 2), " at 150:", round(gap8_big, 2))
assert gap8_small > 3*gap8_big and curves[1][0][-1] - SIG**2/2 > 0.5; print("OK: more data closes the variance gap; degree 1 stays above the floor")

## Task 7: Error functions in code (from your notes)
Write the classification error as the **fraction of wrong predictions**, both with a loop and vectorised.

In [ ]:
yhat = np.array([1, 0, 1, 1, 0, 0, 1, 0])
ytrue = np.array([1, 0, 0, 1, 0, 1, 1, 0])

wrong = 0
for i in range(len(ytrue)):
    if yhat[i] ___ ytrue[i]:
        wrong += 1
cerr_loop = wrong / len(ytrue)
cerr_vec = np.mean(yhat ___ ytrue)
print(cerr_loop, cerr_vec)
assert cerr_loop == cerr_vec == 0.25; print("OK")

## Task 8: Your own project (written answer)
Compute J_train and J_cv for your project model. State your baseline. Which gap is bigger? Which fix from slide 11 will you try first, and why?

In [ ]:
my_diagnosis = "___"
print(my_diagnosis)

## Bonus (optional, needs a few minutes): does transfer learning help?
We tried this on scikit-learn digits: pretrain a small network on digits 0 to 4, then reuse its layers for digits 5 to 9 with only 30 training pictures. The result was **worse** than training from scratch (0.910 scratch, 0.630 frozen layers, 0.502 fine-tuned; mean of 6 seeds, 866 test pictures). Why do you think so? Try pretraining on digits 0 to 9 and transferring to a different kind of image data. Write what you see.

## Wrap-up
- Three sets: train to fit, cross-validation to choose, test once at the end.
- Bias gap = J_train − baseline; variance gap = J_cv − J_train. Fix what is bigger.
- λ and the degree are chosen on J_cv; learning curves tell you if more data will help.
- Next: images and convolution (CNNs).